# Power Spectral Density

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dirs = {
    'vae_transpose':  Path("output/L16B06"), 
    'vae_bilinear':   Path("output/BILINEAR"), 
}

## Load test dataset

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
test  = ds['tephra_col_mass']

## Data map

In [ ]:
## Data map
data_map = {
    'test': test.values,
    'vae_transpose':  None,
    'vae_bilinear': None
}

## Generate a VAE ensemble

In [ ]:
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

In [ ]:
for key, output_dir in output_dirs.items():
    ## Load weight parameters and some metadata
    fname = output_dir / 'model.pt'
    checkpoint = torch.load(fname)

    checkpoint.setdefault("UPSAMPLING", "transpose")

    ## Recreate the model
    model = VariationalAutoencoder(checkpoint['LATENT_DIM'], upsampling=checkpoint['UPSAMPLING'])
    model.load_state_dict(checkpoint['model_state_dict'])

    ## Normalization
    min_value = checkpoint['MINVAL']
    max_value = checkpoint['MAXVAL']
    transform = MinMaxScale(min_value, max_value)
    
    ## Generate nens new samples
    nens = 5000
    z = torch.randn(nens, checkpoint['LATENT_DIM'])
    with torch.no_grad():
        new_sample = model.decode(z)
        x = transform.invert(new_sample).squeeze()
    data_map[key] = x.numpy()

## Compute spectrum

In [ ]:
from modules.metrics import isotropic_spectrum_ensemble

pk_map = {key: isotropic_spectrum_ensemble(value) for key, value in data_map.items()}

## Plot configuration

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 200

DEFAULTS = dict(
    label = None,
    color = 'r',
    ls = 'None',
    alpha = 1.0,
    marker = None,
)

conf_all = {
    'test': dict(label='Test dataset', ls='solid'),
    'vae_transpose':  dict(label='VAE-transposed convolution', alpha=0.5, color = 'g', marker='^'),
    'vae_bilinear': dict(label='VAE-bilinear interpolation', alpha=0.4, color = 'b', marker='+'),   
}

## Plot PSD

In [ ]:
fig, ax = plt.subplots()

ax.plot([0.333,0.333],[0,1E-6], 'k--', alpha=0.4)
for key, value in pk_map.items():
    conf = DEFAULTS | conf_all[key]
    k, Pk, _, _ = value
    ax.plot(k,Pk,
            color  = conf['color'],
            label  = conf['label'],
            alpha  = conf['alpha'],
            marker = conf['marker'],
            ls     = conf['ls'],
           )

ax.set(xlabel=r'Wavenumber, k [$1/\Delta x$]',
       ylabel=r'Isotropic power spectrum, P(k) [$g^2/m^4$]',
       yscale='log')

ax.annotate(r'$\sim 3\Delta x$'+'\nThree grid cells', xy=(0.333, 1E-6), xycoords='data',
            xytext=(0.25, .4), textcoords='axes fraction',
            va='top', ha='center',
            fontsize = 14,
            arrowprops=dict(arrowstyle="->", connectionstyle="arc3"))
ax.legend()

## Compute power fraction for small structures

In [ ]:
cutoff = 0.33

for key, (k_centers, E_mean, E_std, n_modes) in pk_map.items():
    E_power = E_mean * n_modes

    high_k = k_centers > cutoff

    total_power = E_power.sum()
    high_k_power = E_power[high_k].sum()
    fraction = high_k_power / total_power

    print(
        f"{key}: "
        f"high-k power = {high_k_power:.6e}, "
        f"fraction = {fraction:.5%}"
    )

In [ ]:
(20.43-9.21)/20.43